[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C56_Detection_Augmentation_Course/01_geometric/01_geometric_aug.ipynb)

# 01 · 几何增强与标注同步（仿射矩阵 / 四角法 / 框膨胀 / 越界三分法 / letterbox 逆变换 / 翻转白名单）

纯 numpy + 标准库，CPU 可跑，不联网。

**核心命题**：几何增强改的是坐标，而标注也是坐标——所以**两条轨道必须用同一个变换对象**。
本 notebook 把这条纪律落成可执行的代码与断言。

你会亲手实现：

1. **齐次仿射矩阵**（平移/缩放/旋转/错切/透视）与「绕某点变换」的三明治结构
2. **四角法 `warp_boxes`**，以及**无条件的透视除法**（忘了它就是透视增强的头号 bug）
3. **旋转导致的框膨胀**：闭式公式、数值验证、以及**从可接受膨胀率反解最大旋转角**
4. **四角法对圆形标志的过估计**：45° 时标注 IoU 只剩 0.50——这是算法引入的错标注
5. **越界三分法 keep / ignore / drop**，以及 `min_side` 如何单方面锁死最大检出距离
6. **letterbox 正逆变换**，复现 4 种真实世界的逆变换 bug 并量化像素误差
7. **翻转三问判定 + TSR 白名单**，量化「整图可翻率」与「对称类信息增益为零」
8. **RandomIoUCrop** 的 zoom-in 收益与**位置先验被摧毁**的代价

> 心智模型：**bbox 不是图像的一部分，它是图像的一个「投影」。
> 变换图像时，你必须把这个投影重新算一遍——而 min/max 会让它每次都变得更差一点。**

## 1 · 齐次坐标与仿射矩阵

把所有几何变换合成**一个 3×3 矩阵**，不是为了代码短，而是为了三件事：
① 只插值一次（三次双线性插值足以糊掉 15 px 标志上的限速数字）；
② 「绕某点变换」有明确写法；③ 图像与标注**用的是同一个对象**，不可能不同步。

In [ ]:
import numpy as np, math, collections, itertools
np.set_printoptions(suppress=True, precision=3)
rng = np.random.default_rng(7)

# ── 基本矩阵（齐次坐标，点写成列向量 [x, y, 1]^T）──
def M_T(tx, ty):
    return np.array([[1, 0, tx], [0, 1, ty], [0, 0, 1]], float)

def M_S(sx, sy=None):
    sy = sx if sy is None else sy
    return np.array([[sx, 0, 0], [0, sy, 0], [0, 0, 1]], float)

def M_R(deg):
    t = math.radians(deg); c, s = math.cos(t), math.sin(t)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]], float)

def M_H(deg_x, deg_y=0.0):                       # 错切 shear
    return np.array([[1, math.tan(math.radians(deg_x)), 0],
                     [math.tan(math.radians(deg_y)), 1, 0],
                     [0, 0, 1]], float)

def M_P(px, py=0.0):                             # 透视：第三行不再是 [0,0,1]
    return np.array([[1, 0, 0], [0, 1, 0], [px, py, 1]], float)

def about(M, cx, cy):
    # **绕 (cx,cy) 做变换** = 移到原点 -> 变换 -> 移回去。手写仿射最常错的地方。
    return M_T(cx, cy) @ M @ M_T(-cx, -cy)

def compose(*Ms):
    out = np.eye(3)
    for M in Ms:
        out = out @ M                            # **从右往左作用**：最右边的先作用在点上
    return out

# ── 顺序是语义的一部分：矩阵乘法不可交换 ──
A = compose(M_T(100, 0), M_S(0.5))               # 先缩放，再平移
B = compose(M_S(0.5), M_T(100, 0))               # 先平移，再缩放
pa = (A @ np.array([0., 0., 1.]))[:2]
pb = (B @ np.array([0., 0., 1.]))[:2]
print('原点 (0,0) 经过')
print(f'  先缩放0.5 再平移100 -> {pa}')
print(f'  先平移100 再缩放0.5 -> {pb}')
assert not np.allclose(A, B), '矩阵乘法不可交换'
assert abs(pa[0] - pb[0] - 50) < 1e-9, '两种顺序差 50 像素'

# ── 「绕中心旋转」的不动点 ──
Mc = about(M_R(90), 5, 5)
fixed = (Mc @ np.array([5., 5., 1.]))[:2]
corner = (M_R(90) @ np.array([5., 5., 1.]))[:2]  # 绕原点转：中心被甩走了
print(f'\n绕(5,5)转90度: (5,5) -> {fixed}   (不动点 ✅)')
print(f'绕原点转90度  : (5,5) -> {corner}   (整幅图被甩出画面 ❌)')
assert np.allclose(fixed, [5., 5.])
assert not np.allclose(corner, [5., 5.])
print('\n✅ 矩阵工具就位。记住 T(c) · M · T(-c) 这个三明治结构。')

## 2 · 四角法 `warp_boxes`：bbox 跟随任意变换

轴对齐 bbox 在旋转/错切/透视下**不是封闭的**，所以统一做法是：
**变换 4 个角点 → 取轴对齐外接框**。

注意 `apply_M` 里**无条件做透视除法** `x' = x_h / w_h`：
仿射时 `w_h = 1`，这是恒等操作；透视时它是必需的。
**忘记这一步是透视增强的头号 bug，且幅度小的时候误差也小，能潜伏很久。**

In [ ]:
def apply_M(M, pts):
    pts = np.asarray(pts, float).reshape(-1, 2)
    h = np.concatenate([pts, np.ones((len(pts), 1))], axis=1) @ M.T
    return h[:, :2] / h[:, 2:3]                  # ← 无条件透视除法

def apply_M_naive(M, pts):                       # ❌ 忘记透视除法的错误版本
    pts = np.asarray(pts, float).reshape(-1, 2)
    h = np.concatenate([pts, np.ones((len(pts), 1))], axis=1) @ M.T
    return h[:, :2]

def warp_boxes(boxes, M):
    boxes = np.asarray(boxes, float).reshape(-1, 4)
    if len(boxes) == 0:
        return boxes.copy()
    x1, y1, x2, y2 = boxes.T
    corners = np.stack([x1, y1, x2, y1, x2, y2, x1, y2], axis=1).reshape(-1, 2)
    p = apply_M(M, corners).reshape(len(boxes), 4, 2)
    return np.concatenate([p.min(axis=1), p.max(axis=1)], axis=1)

def box_area(b):
    b = np.asarray(b, float).reshape(-1, 4)
    return np.clip(b[:, 2] - b[:, 0], 0, None) * np.clip(b[:, 3] - b[:, 1], 0, None)

def box_iou(a, b):
    a = np.asarray(a, float).reshape(-1, 4); b = np.asarray(b, float).reshape(-1, 4)
    lt = np.maximum(a[:, None, :2], b[None, :, :2])
    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    wh = np.clip(rb - lt, 0, None)
    inter = wh[..., 0] * wh[..., 1]
    return inter / np.maximum(box_area(a)[:, None] + box_area(b)[None, :] - inter, 1e-12)

BOX = np.array([[0., 0., 40., 40.]])             # 一块 40x40 的方框（圆形标志的外接框）
print('恒等   ', warp_boxes(BOX, np.eye(3))[0])
print('平移+10,-5', warp_boxes(BOX, M_T(10, -5))[0])
print('缩放 2x ', warp_boxes(BOX, M_S(2))[0])
print('绕中心转90度', warp_boxes(BOX, about(M_R(90), 20, 20))[0], '  <- 正方形转 90 度：不变')
assert np.allclose(warp_boxes(BOX, np.eye(3)), BOX)
assert np.allclose(warp_boxes(BOX, M_T(10, -5))[0], [10, -5, 50, 35])
assert np.allclose(warp_boxes(BOX, M_S(2))[0], [0, 0, 80, 80])
assert np.allclose(warp_boxes(BOX, about(M_R(90), 20, 20)), BOX)

RECT = np.array([[0., 0., 20., 10.]])            # 非正方形：转 90 度 -> 宽高互换，面积不变
w90 = warp_boxes(RECT, about(M_R(90), 10, 5))[0]
print('\n20x10 框绕中心转90度 ->', w90, ' 面积', box_area([w90])[0], '(原', box_area(RECT)[0], ')')
assert np.allclose(box_area([w90]), box_area(RECT))

# ── 忘记透视除法的后果 ──
Mp = M_P(2e-4, 0.0)                              # 一个很温和的透视
pt = np.array([[640., 320.]])
ok, bad = apply_M(Mp, pt)[0], apply_M_naive(Mp, pt)[0]
err = float(np.abs(ok - bad).max())
print(f'\n透视 px=2e-4，点 (640,320):  正确 {ok}   忘记除法 {bad}   误差 {err:.1f} px')
assert err > 50, '忘记透视除法在图像边缘会造成几十像素的误差'
print('⚠️  幅度小时误差也小（w≈1），所以这个 bug 能在代码库里潜伏很久 ——')
print('    直到有人把 perspective 从 0.0005 调到 0.005，框开始莫名其妙地偏。')
print('✅ 对策：角点变换函数**无条件**做透视除法，仿射时它只是恒等操作。')

## 3 · 旋转导致的框膨胀：算到小数点后

设原框 `w × h`，绕中心旋转 `θ`，四角外接框的宽高是
`w|cosθ| + h|sinθ|` 与 `w|sinθ| + h|cosθ|`。展开相乘可得一个很干净的闭式：

$$\frac{A'}{A} = 1 + \frac{1}{2}\,|\sin 2\theta|\left(\frac{w}{h} + \frac{h}{w}\right)$$

三个推论：**膨胀在 45° 最大（不是 90°）**；**正方形框膨胀最小**（45° 时正好 2.00×）；
**细长框膨胀更狠**（3:1 的指路牌 45° 时 2.67×）。

In [ ]:
def inflate_ratio(w, h, deg):
    t = math.radians(deg); c, s = abs(math.cos(t)), abs(math.sin(t))
    return ((w * c + h * s) * (w * s + h * c)) / (w * h)

def inflate_closed(w, h, deg):
    return 1 + 0.5 * abs(math.sin(2 * math.radians(deg))) * (w / h + h / w)

# 闭式 vs 展开式 vs 真实 warp_boxes：三者必须一致
for deg in range(0, 181, 5):
    assert abs(inflate_ratio(40, 40, deg) - inflate_closed(40, 40, deg)) < 1e-9
    assert abs(inflate_ratio(60, 20, deg) - inflate_closed(60, 20, deg)) < 1e-9
    got = box_area(warp_boxes(BOX, about(M_R(deg), 20, 20)))[0] / box_area(BOX)[0]
    assert abs(got - inflate_ratio(40, 40, deg)) < 1e-9, deg
print('✅ 闭式公式 / 展开式 / warp_boxes 实测，三者在 0-180 度上完全一致')

print(f"\n{'θ':>5s} {'正方形40x40':>12s} {'3:1指路牌':>11s} {'圆形标志框内前景占比':>20s} {'与真实紧框IoU':>14s}")
for deg in [0, 5, 10, 15, 30, 45, 60, 90]:
    t = math.radians(deg); c, s = abs(math.cos(t)), abs(math.sin(t))
    fg = (math.pi / 4) / (c + s) ** 2            # 圆内接方框，旋转后四角框被撑大
    iou = 1.0 / (c + s) ** 2                     # 四角框 vs 真实紧框（圆是旋转不变的）
    print(f'{deg:>4d}° {inflate_ratio(40,40,deg):>11.3f}x {inflate_ratio(60,20,deg):>10.3f}x '
          f'{fg:>19.1%} {iou:>14.3f}')

assert abs(inflate_ratio(40, 40, 45) - 2.0) < 1e-9,  '正方形框 45 度 -> 面积翻倍'
assert abs(inflate_ratio(40, 40, 15) - 1.5) < 1e-9,  '正方形框 15 度 -> 已膨胀 50%'
assert abs(inflate_ratio(60, 20, 45) - 8/3) < 1e-9,  '3:1 框 45 度 -> 2.667x'
assert inflate_ratio(40, 40, 90) < inflate_ratio(40, 40, 45), '最糟的是 45 度，不是 90 度'
print('\n⚠️  仅仅 15 度，正方形框的面积就已经膨胀 50% —— 多出来的全是背景。')
print('    模型被反复告知「框住一半背景也算对」，定位头就这样被慢慢教松：')
print('    典型指纹是 **AP50 正常但 AP75 明显偏低**。')

# ── 从「可接受的膨胀率」反解最大旋转角（写进配置评审清单）──
def max_rot_angle(w, h, max_inflate):
    k = (max_inflate - 1) / (0.5 * (w / h + h / w))
    return 45.0 if k >= 1 else math.degrees(math.asin(k)) / 2

print(f"\n可接受膨胀 <=35% 时的旋转角上限：")
for (w, h, name) in [(40, 40, '正方形（圆形/方形标志）'), (60, 30, '2:1'), (60, 20, '3:1 横向指路牌')]:
    print(f'  {name:<22s} |θ| <= {max_rot_angle(w, h, 1.35):5.2f}°')
assert abs(max_rot_angle(40, 40, 1.35) - 10.24) < 0.05
assert abs(max_rot_angle(60, 20, 1.35) - 6.06) < 0.05
assert max_rot_angle(60, 20, 1.35) < max_rot_angle(40, 40, 1.35), '细长框上限更严'
print('\n✅ 「我不是选了个 ±10°，我是从可接受的标注退化反推出来的」—— 面试答案的正确形状。')

In [ ]:
# ── 四角法的过估计：对**旋转不变**的形状（圆形禁令牌！）它给出的框是**错的** ──
def circle_pts(cx, cy, r, n=512):
    a = np.linspace(0, 2 * np.pi, n, endpoint=False)
    return np.stack([cx + r * np.cos(a), cy + r * np.sin(a)], axis=1)

def ngon_pts(cx, cy, r, k, rot_deg=0.0):
    a = np.linspace(0, 2 * np.pi, k, endpoint=False) + math.radians(rot_deg)
    return np.stack([cx + r * np.cos(a), cy + r * np.sin(a)], axis=1)

def tight_bbox(pts):
    pts = np.asarray(pts, float)
    return np.array([pts[:, 0].min(), pts[:, 1].min(), pts[:, 0].max(), pts[:, 1].max()])

def poly_area(pts):                              # shoelace
    x, y = np.asarray(pts, float).T
    return 0.5 * abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))

SHAPES = {
    '圆形禁令牌(旋转不变)': circle_pts(20, 20, 20),
    '三角警告牌':          ngon_pts(20, 20, 20, 3, rot_deg=-90),
    '方形指示牌(填满框)':   np.array([[0., 0.], [40., 0.], [40., 40.], [0., 40.]]),
}
print(f"{'形状':<22s} {'θ':>4s} {'四角框':>28s} {'真实紧框':>28s} {'标注IoU':>8s} {'框内前景':>9s}")
for name, pts in SHAPES.items():
    orig = tight_bbox(pts)
    cx, cy = (orig[0] + orig[2]) / 2, (orig[1] + orig[3]) / 2
    for deg in [0, 45]:
        M = about(M_R(deg), cx, cy)
        four = warp_boxes([orig], M)[0]
        wpts = apply_M(M, pts)
        true = tight_bbox(wpts)
        iou = box_iou([four], [true])[0, 0]
        fg = poly_area(wpts) / max(box_area([four])[0], 1e-9)
        print(f'{name:<22s} {deg:>3d}° {np.round(four,1)} {np.round(true,1)} {iou:>8.3f} {fg:>9.1%}')
        # 四角框**总是包含**真实紧框（凸形状的性质）
        assert four[0] <= true[0] + 1e-6 and four[1] <= true[1] + 1e-6
        assert four[2] >= true[2] - 1e-6 and four[3] >= true[3] - 1e-6

M45 = about(M_R(45), 20, 20)
c_four = warp_boxes([tight_bbox(SHAPES['圆形禁令牌(旋转不变)'])], M45)[0]
c_true = tight_bbox(apply_M(M45, SHAPES['圆形禁令牌(旋转不变)']))
c_iou = box_iou([c_four], [c_true])[0, 0]
c_fg = poly_area(apply_M(M45, SHAPES['圆形禁令牌(旋转不变)'])) / box_area([c_four])[0]
assert abs(c_iou - 0.5) < 0.01, f'圆形标志 45 度：四角框与真实紧框 IoU 应为 0.50，实得 {c_iou:.3f}'
assert abs(c_fg - math.pi / 8) < 0.01, f'框内前景应为 π/8 = 39.3%，实得 {c_fg:.3f}'

# 对比：**恰好填满框**的形状，四角法是精确的
s_four = warp_boxes([tight_bbox(SHAPES['方形指示牌(填满框)'])], M45)[0]
s_true = tight_bbox(apply_M(M45, SHAPES['方形指示牌(填满框)']))
assert np.allclose(s_four, s_true), '目标填满框时，四角法精确'
print(f'\n⚠️  圆形禁令牌旋转 45°：真实紧框**一点没变**（圆是旋转不变的），')
print(f'    四角法却把框撑到 √2 倍边长 —— 标注 IoU 只剩 {c_iou:.2f}，框内前景只剩 {c_fg:.1%}。')
print('    **这不是「任务变难了」（那是好事），这是「标签变错了」（那是坏事）。**')
print('✅ 而对「恰好填满框」的目标（车、行人），四角法是精确的 —— 差别就在这里。')

## 4 · 越界处理：keep / ignore / drop

一句话区分：**`keep` 说「有」，`ignore` 说「不知道」，`drop` 说「没有」**。

大多数实现的默认行为是 `drop`（一行 `boxes = boxes[mask]` 就写完了），
于是「可见比例 40% 的目标被删掉、而图像里那 40% 还清清楚楚留着」成了默认行为——
**模型被明确训练成「看到半个标志 = 这里没有东西」。**

In [ ]:
def clip_split(boxes, labels, W, H, min_visible=0.5, min_side=2.0):
    boxes = np.asarray(boxes, float).reshape(-1, 4)
    labels = np.asarray(labels)
    a0 = np.clip(boxes[:, 2] - boxes[:, 0], 0, None) * np.clip(boxes[:, 3] - boxes[:, 1], 0, None)
    c = boxes.copy()
    c[:, [0, 2]] = np.clip(c[:, [0, 2]], 0, W)
    c[:, [1, 3]] = np.clip(c[:, [1, 3]], 0, H)
    w = np.clip(c[:, 2] - c[:, 0], 0, None); h = np.clip(c[:, 3] - c[:, 1], 0, None)
    vis = np.where(a0 > 0, (w * h) / np.maximum(a0, 1e-9), 0.0)
    keep = (vis >= min_visible) & (w >= min_side) & (h >= min_side)
    drop = vis <= 0                              # 完全出画面 -> 真的没有东西
    ignore = (~keep) & (~drop)                   # 仍可见但给不出可靠框 -> 别学
    return {'keep': keep, 'ignore': ignore, 'drop': drop, 'vis': vis, 'clipped': c}

W = H = 640
DEMO = np.array([
    [100., 100., 140., 140.],     # 完全在内
    [-20., 100.,  20., 140.],     # 左边切掉一半
    [-30., 100.,  10., 140.],     # 左边切掉 3/4
    [600., 300., 660., 360.],     # 右边切掉 1/3
    [660., 300., 700., 360.],     # 完全在外
    [300.,  -4., 304.,   4.],     # 上边界，小目标，露一半
    [500., 500., 501.5, 530.],    # 极细的框（宽 1.5 px）
    [-50., -50., -10., -10.],     # 完全在外
])
LBL = np.arange(len(DEMO))
r = clip_split(DEMO, LBL, W, H, min_visible=0.5, min_side=2.0)
print(f"{'#':>2s} {'原框':>28s} {'可见比例':>8s} {'裁剪后':>28s}  归属")
for i in range(len(DEMO)):
    tag = 'keep' if r['keep'][i] else ('drop' if r['drop'][i] else 'IGNORE')
    print(f'{i:>2d} {np.round(DEMO[i],1)} {r["vis"][i]:>8.2f} {np.round(r["clipped"][i],1)}  {tag}')

assert r['keep'].sum() == 4 and r['ignore'].sum() == 2 and r['drop'].sum() == 2
assert r['ignore'][2] and r['ignore'][6], '可见比例不足、或框过细 -> ignore（不是 drop）'
assert r['drop'][4] and r['drop'][7], '只有完全出画面才 drop'
assert abs(r['vis'][1] - 0.5) < 1e-12 and r['keep'][1], 'vis 恰好等于阈值应保留'
print('\n✅ 只有 vis == 0（完全出画面）才 drop。**其余一律 ignore，绝不删除。**')
print('   mmdet 的 gt_bboxes_ignore / COCO 的 iscrowd 就是干这个用的。')

In [ ]:
# ── min_side 单方面锁死了整个系统的最大检出距离（针孔模型）──
def focal_px(img_w, hfov_deg):
    return img_w / (2 * math.tan(math.radians(hfov_deg) / 2))

def sign_px(Z, img_w=1920, hfov=60.0, S=0.6, input_w=None):
    px = focal_px(img_w, hfov) * S / Z
    return px if input_w is None else px * input_w / img_w

def max_detect_range(min_side_px, input_w=640, sensor_w=1920, hfov_deg=60.0, sign_m=0.6):
    return focal_px(sensor_w, hfov_deg) * sign_m * (input_w / sensor_w) / min_side_px

f = focal_px(1920, 60)
print(f'相机: 1920px 宽, 60° HFOV -> 焦距 {f:.1f} px；标志物理尺寸 0.6 m；训练输入 640')
print(f"\n{'距离':>6s} {'全分辨率像素':>13s} {'640输入下像素':>14s}")
for Z in [20, 40, 60, 80, 100]:
    print(f'{Z:>4d} m {sign_px(Z):>12.1f} {sign_px(Z, input_w=640):>14.1f}')
assert abs(sign_px(60) - 16.63) < 0.05 and abs(sign_px(60, input_w=640) - 5.54) < 0.05

SPEED = 120 / 3.6                                # 120 km/h -> m/s
print(f"\n{'min_side':>9s} {'最大检出距离':>13s} {'120km/h 下的反应时间':>21s}  判断")
VERDICT = {8: '❌ 远处标志在标注阶段就被删光了', 4: '⚠️ 勉强；4px 框标注误差已达 25%',
           2: '✅ YOLOv5 默认值，TSR 的合理下限', 1: '⚠️ 已无实际意义，但不再人为设限'}
for ms in [8, 4, 2, 1]:
    Zmax = max_detect_range(ms)
    print(f'{ms:>7d}px {Zmax:>11.1f} m {Zmax/SPEED:>18.1f} s  {VERDICT[ms]}')

assert abs(max_detect_range(8) - 41.57) < 0.05, max_detect_range(8)
assert abs(max_detect_range(2) - 166.28) < 0.05
assert max_detect_range(2) == 4 * max_detect_range(8)
print('\n⚠️  一个「看起来无害」的 min_side=8，把最大检出距离锁死在 41.6 米 ——')
print('    高速上只剩 1.2 秒反应时间。**模型再强也没用，样本在标注阶段就没了。**')
print('✅ 排查入口：把增强前后的目标尺寸直方图画出来对比，看小尺寸桶是不是整段消失了。')

In [ ]:
# ── 「drop 而非 ignore」制造了多少假负样本监督 ──
N_SIGNS, P_BORDER = 20000, 0.18                  # TSR 里约 18% 的标志与画面边缘相交
g = np.random.default_rng(2024)
touches = g.random(N_SIGNS) < P_BORDER
vis = np.ones(N_SIGNS)
vis[touches] = g.uniform(0.05, 1.0, touches.sum())   # 被截断的那部分，可见比例近似均匀

print(f"{'min_visible':>11s} {'保留(正样本)':>13s} {'drop策略下变成背景的**可见**目标':>32s} {'ignore策略':>11s}")
bg_counts = {}
for mv in [0.1, 0.25, 0.5, 0.9]:
    keep = vis >= mv
    bg = int(((~keep) & (vis > 0)).sum())        # 仍然可见、却被当成背景
    bg_counts[mv] = bg
    print(f'{mv:>11.2f} {int(keep.sum()):>13d} {bg:>26d} ({bg/N_SIGNS:>5.1%}) {0:>11d}')

assert bg_counts[0.9] > bg_counts[0.5] > bg_counts[0.25] > bg_counts[0.1]
assert bg_counts[0.5] > 0.06 * N_SIGNS, '阈值 0.5 时，超过 6% 的目标变成了假负样本'
per_1k = bg_counts[0.5] / N_SIGNS * 1000 * 2.5   # 按每图约 2.5 个标志换算
print(f'\n⚠️  min_visible=0.5 + drop：每 1000 张训练图里，约有 {per_1k:.0f} 个**仍然可见**的标志')
print('    被当成了背景。而 TSR 里「目标被画面边缘截断」不是异常，是每次接近标志的必经阶段：')
print('    标志从画面上方进入视野，先露下半截，再逐渐完整。')
print('    症状：**首次检出距离系统性偏近 + 图像上边缘召回明显低于中心** ——')
print('    而这两个都在标准 mAP 里看不见（要靠 C55 模块 05 的分桶评测才显形）。')
print('✅ ignore 策略把这个数字变成 0，代价只是少了一点点监督。**没有理由不用。**')

## 5 · letterbox：正变换、逆变换与四种真实 bug

`letterbox` 不是增强，是输入尺寸归一化——但它同样改坐标，
而且是**全流水线里唯一必须与推理端逐像素一致的算子**。

**逆变换是它存在的全部理由**：网络输出在 640×640 的 letterbox 坐标系里，
下游（跟踪 / 融合 / 地图匹配 / VLA 接口）要的是原图坐标。这一步写错，模型再准也白搭。
C60 模块 04 把「letterbox 逆变换写错」列为「框整体偏移」类问题的头号成因。

In [ ]:
def letterbox_params(src_hw, dst_hw, center=True, scaleup=False, stride=None, round_fn=None):
    round_fn = round if round_fn is None else round_fn
    (Hs, Ws), (Ht, Wt) = src_hw, dst_hw
    r = min(Ht / Hs, Wt / Ws)
    if not scaleup:
        r = min(r, 1.0)                          # 推理时通常不放大小图
    nw, nh = int(round_fn(Ws * r)), int(round_fn(Hs * r))
    dw, dh = Wt - nw, Ht - nh
    if stride:
        dw, dh = dw % stride, dh % stride        # auto=True：只 pad 到 stride 倍数
    left = dw / 2 if center else 0.0
    top = dh / 2 if center else 0.0
    return {'r': r, 'new': (nh, nw), 'left': left, 'top': top, 'out': (nh + dh, nw + dw)}

def lb_fwd(boxes, p):
    b = np.asarray(boxes, float).reshape(-1, 4).copy()
    b[:, [0, 2]] = b[:, [0, 2]] * p['r'] + p['left']
    b[:, [1, 3]] = b[:, [1, 3]] * p['r'] + p['top']
    return b

def lb_inv(boxes, r, left, top, ry=None):
    b = np.asarray(boxes, float).reshape(-1, 4).copy()
    b[:, [0, 2]] = (b[:, [0, 2]] - left) / r
    b[:, [1, 3]] = (b[:, [1, 3]] - top) / (ry if ry else r)
    return b

SRC, DST = (1080, 1920), (640, 640)
VARIANTS = [
    ('默认（居中 + round）',      dict()),
    ('右下填充（torchvision式）',  dict(center=False)),
    ('auto=True（pad 到 32 倍数）', dict(stride=32)),
    ('floor 取整（C++ 常见）',     dict(round_fn=math.floor)),
]
print(f"{'变体':<26s} {'r':>7s} {'缩放后(h,w)':>13s} {'(left,top)':>15s} {'输出(h,w)':>13s}")
PS = {}
for name, kw in VARIANTS:
    p = letterbox_params(SRC, DST, **kw); PS[name] = p
    print(f'{name:<26s} {p["r"]:>7.4f} {str(p["new"]):>13s} '
          f'{str((p["left"], p["top"])):>15s} {str(p["out"]):>13s}')

p_ok = PS['默认（居中 + round）']
assert p_ok['new'] == (360, 640) and p_ok['top'] == 140.0 and p_ok['left'] == 0.0
assert PS['右下填充（torchvision式）']['top'] == 0.0, '右下填充 -> 整体偏 140/r = 420 px'
assert PS['auto=True（pad 到 32 倍数）']['out'] == (384, 640), '**导出 ONNX/TRT 时必须 auto=False**'

# ⚠️ 取整分歧在 1920x1080 上**看不出来**：1080*(640/1920) 恰好是 360.0
assert PS['floor 取整（C++ 常见）']['top'] == p_ok['top'], '整除时 round 与 floor 一致 —— 所以开发机上永远复现不了'
ROI = (800, 1920)                                # 车端常见：切掉天空与引擎盖后的 ROI
p_roi_r = letterbox_params(ROI, DST)
p_roi_f = letterbox_params(ROI, DST, round_fn=math.floor)
print(f'\n换成车端 ROI 裁剪图 1920x800（800*(640/1920) = {800*p_roi_r["r"]:.4f}，不整除）：')
print(f'  round -> new={p_roi_r["new"]}, top={p_roi_r["top"]}')
print(f'  floor -> new={p_roi_f["new"]}, top={p_roi_f["top"]}   <- **差了半个 padding 像素**')
assert p_roi_r['new'] == (267, 640) and p_roi_f['new'] == (266, 640)
assert abs(p_roi_f['top'] - p_roi_r['top'] - 0.5) < 1e-12

TRUE24 = np.array([[948., 528., 972., 552.]])    # 画面中心一块 24px 的标志
lb = lb_fwd(TRUE24, p_ok)
print(f'\n原图框 {TRUE24[0]}  --letterbox-->  {lb[0]}   (24px -> 8px)')
assert np.allclose(lb, [[316., 316., 324., 324.]])
assert np.allclose(lb_inv(lb, p_ok['r'], p_ok['left'], p_ok['top']), TRUE24), '正逆往返必须无损'
print('✅ 正逆变换往返无损。padding 占了 640x640 的 %.1f%% —— 这些网格点永远是负样本。'
      % (100 * (640 * 640 - 640 * 360) / (640 * 640)))

In [ ]:
# ── 三种「灾难性」的逆变换 bug（1920x1080）──
r_, left_, top_ = p_ok['r'], p_ok['left'], p_ok['top']
rx, ry = 640 / 1920, 640 / 1080                  # 各轴独立缩放（错的）
p_str = PS['auto=True（pad 到 32 倍数）']
TRUE8 = np.array([[956., 536., 964., 544.]])     # 同一位置的 8px 远处标志

CASES = [
    ('✅ 正确',                    p_ok,  lambda b: lb_inv(b, r_, left_, top_)),
    ('① 忘记减 padding',           p_ok,  lambda b: lb_inv(b, r_, 0.0, 0.0)),
    ('② 各轴独立缩放 + 无 pad',     p_ok,  lambda b: lb_inv(b, rx, 0.0, 0.0, ry)),
    ('③ 前向 auto=True / 逆用 640', p_str, lambda b: lb_inv(b, r_, left_, top_)),
]
print(f"{'逆变换实现':<28s} {'还原出的框(24px标志)':>30s} {'y误差px':>9s} {'IoU@24px':>9s} {'IoU@8px':>8s}")
res = {}
for name, p_fwd, inv in CASES:
    rec24 = inv(lb_fwd(TRUE24, p_fwd)); rec8 = inv(lb_fwd(TRUE8, p_fwd))
    i24 = box_iou(rec24, TRUE24)[0, 0]; i8 = box_iou(rec8, TRUE8)[0, 0]
    dy = abs(rec24[0, 1] - TRUE24[0, 1])
    res[name] = (i24, i8, dy)
    print(f'{name:<28s} {np.round(rec24[0],1)} {dy:>9.1f} {i24:>9.3f} {i8:>8.3f}')

assert res['✅ 正确'][0] > 0.9999 and res['✅ 正确'][1] > 0.9999
assert res['① 忘记减 padding'][0] == 0.0 and abs(res['① 忘记减 padding'][2] - 420) < 0.5
assert res['③ 前向 auto=True / 逆用 640'][0] == 0.0
assert abs(res['③ 前向 auto=True / 逆用 640'][2] - 384) < 0.5
assert 0.55 < res['② 各轴独立缩放 + 无 pad'][0] < 0.58
print('\n⚠️  bug ①③ 让框整体偏 420 / 384 像素 —— 灾难性，但**一眼就能看出来**（框飞了）。')

# ── ④ 取整分歧：**只在源尺寸不整除时暴露**，所以最危险 ──
TRUE24_ROI = np.array([[948., 388., 972., 412.]])   # 1920x800 的 ROI 里，一块 24px 标志
TRUE8_ROI  = np.array([[956., 396., 964., 404.]])   # 同一位置的 8px 标志
print(f"\n{'④ 前向 round / 逆用 floor':<28s} {'源尺寸':>12s} {'y误差px':>9s} {'IoU@24px':>9s} {'IoU@8px':>8s}")
for tag, p_f, p_b, t24, t8 in [
        ('  在 1920x1080 上', p_ok,    PS['floor 取整（C++ 常见）'], TRUE24, TRUE8),
        ('  在 1920x800 ROI 上', p_roi_r, p_roi_f,                  TRUE24_ROI, TRUE8_ROI)]:
    rec24 = lb_inv(lb_fwd(t24, p_f), p_f['r'], p_b['left'], p_b['top'])
    rec8 = lb_inv(lb_fwd(t8, p_f), p_f['r'], p_b['left'], p_b['top'])
    i24 = box_iou(rec24, t24)[0, 0]; i8 = box_iou(rec8, t8)[0, 0]
    dy = abs(rec24[0, 1] - t24[0, 1])
    print(f'{tag:<28s} {str(p_f["new"]):>12s} {dy:>9.2f} {i24:>9.3f} {i8:>8.3f}')
    if 'ROI' in tag:
        i24_4, i8_4, dy4 = i24, i8, dy

assert dy4 > 0, 'ROI 上取整分歧才暴露'
assert abs(dy4 - 1.5) < 1e-6, '半个 padding 像素 -> 原图 1.5 px'
assert abs(i24_4 - 0.882) < 0.005 and abs(i8_4 - 0.684) < 0.005, (i24_4, i8_4)
assert i8_4 < i24_4, '**同样的亚像素误差，小目标受伤远大于大目标**'
print(f'\n⚠️  bug ④ 在 1920x1080 上**误差为 0**（1080 恰好整除），在开发机上永远复现不了；')
print(f'    一换成车端的 ROI 裁剪图，就偏 {dy4:.1f} 个原图像素 ——')
print(f'    24px 标志 IoU {i24_4:.2f}（还行），**8px 远处标志 IoU 只剩 {i8_4:.2f}**。')
print('    **这种「只低一点点、还只在特定输入下出现」的 bug 才是真正危险的** —— 它像「模型不够好」。')
print('✅ 正确做法：把 (r, left, top) 作为**元数据**跟着图像传下去，不要在推理端重算。')
print('   YOLOv5 的 scale_boxes(..., ratio_pad=...) 参数存在的唯一理由就是这个。')

## 6 · 翻转白名单：TSR 面试可以主动提的加分点

通用检测里 `fliplr=0.5` 是白送的。**交通标志是人为设计的符号系统**——
它的全部意义就在形状、颜色和**朝向**承载的约定。

**三问判定法**：
1. `Q1` 外观是否左右镜像对称？
2. `Q2` 语义是否含方向性（左/右、顺/逆、单向）？
3. `Q3` 标签集里是否存在它的镜像类别？

→ `Q1 且 非Q2` = **SAFE**；`Q2 且 Q3` = **SWAP**（翻转 + **换标签**）；其余 = **FORBID**。

In [ ]:
def decide_flip(mirror_symmetric, directional, has_mirror_class):
    if mirror_symmetric and not directional:
        return 'SAFE'
    if directional and has_mirror_class:
        return 'SWAP'
    return 'FORBID'

# (key, 中文, 外观左右对称, 语义含方向, 镜像类别)
SIGNS = [
    ('no_entry',            '禁止驶入',      True,  False, None),
    ('no_vehicles',         '禁止通行',      True,  False, None),
    ('no_honking',          '禁止鸣笛',      True,  False, None),
    ('traffic_light_ahead', '注意信号灯',    True,  False, None),
    ('turn_left',           '向左转弯',      False, True,  'turn_right'),
    ('turn_right',          '向右转弯',      False, True,  'turn_left'),
    ('no_left_turn',        '禁止向左转弯',  False, True,  'no_right_turn'),
    ('no_right_turn',       '禁止向右转弯',  False, True,  'no_left_turn'),
    ('keep_left',           '左侧通行',      False, True,  'keep_right'),
    ('keep_right',          '右侧通行',      False, True,  'keep_left'),
    ('curve_left',          '向左急弯路',    False, True,  'curve_right'),
    ('curve_right',         '向右急弯路',    False, True,  'curve_left'),
    ('narrow_left',         '左侧变窄',      False, True,  'narrow_right'),
    ('narrow_right',        '右侧变窄',      False, True,  'narrow_left'),
    ('merge_left',          '左侧合流',      False, True,  None),      # ← 标签集里没有右侧合流
    ('speed_limit_30',      '限速30',        False, False, None),
    ('speed_limit_60',      '限速60',        False, False, None),
    ('speed_limit_120',     '限速120',       False, False, None),
    ('stop',                '停车让行STOP',  False, False, None),
    ('guide_sign',          '指路牌',        False, False, None),
    ('ped_crossing',        '注意行人',      False, False, None),
    ('children',            '注意儿童',      False, False, None),
]
TABLE_ = {k: {'zh': zh, 'kind': decide_flip(sym, d, m is not None), 'mirror': m}
          for k, zh, sym, d, m in SIGNS}

by_kind = collections.defaultdict(list)
for k, v in TABLE_.items():
    by_kind[v['kind']].append(v['zh'])
for kind in ['SAFE', 'SWAP', 'FORBID']:
    print(f'{kind:<7s} ({len(by_kind[kind]):2d} 类)  ' + '、'.join(by_kind[kind]))

assert TABLE_['turn_left']['kind'] == 'SWAP' and TABLE_['turn_left']['mirror'] == 'turn_right'
assert TABLE_['no_entry']['kind'] == 'SAFE'
assert TABLE_['speed_limit_60']['kind'] == 'FORBID', '含数字：镜像后的字形现实中不存在'
assert TABLE_['guide_sign']['kind'] == 'FORBID', '指路牌含地名文字'
assert TABLE_['merge_left']['kind'] == 'FORBID', '**Q3 不过**：有方向语义但标签集里没有镜像类'
assert (len(by_kind['SAFE']), len(by_kind['SWAP']), len(by_kind['FORBID'])) == (4, 10, 8)
print('\n⚠️  注意「左侧合流」：它有方向语义、外观也不对称，但**标签集里没有「右侧合流」**，')
print('    翻了就没有正确标签可给 -> FORBID。**Q3 不是走过场，它真的会改变结论。**')

In [ ]:
# ── 整图翻转的实现：逐目标判定，但作用在整图上（「与」的关系）──
def hflip_boxes(boxes, W):
    b = np.asarray(boxes, float).reshape(-1, 4).copy()
    x1, x2 = b[:, 0].copy(), b[:, 2].copy()
    b[:, 0], b[:, 2] = W - x2, W - x1            # 注意 x1/x2 会互换
    return b

def flip_scene(W, boxes, labels, table=None):
    table = TABLE_ if table is None else table
    kinds = [table[c]['kind'] for c in labels]
    if 'FORBID' in kinds:
        return None                              # 一个 FORBID -> 整张图都不能翻
    new = [table[c]['mirror'] if table[c]['kind'] == 'SWAP' else c for c in labels]
    return hflip_boxes(boxes, W), new

for scene in [(['no_entry', 'no_vehicles'],), (['turn_left', 'no_entry'],),
              (['turn_left', 'speed_limit_60'],), (['merge_left'],)]:
    labs = scene[0]
    bx = np.array([[10. + 30 * i, 20., 40. + 30 * i, 50.] for i in range(len(labs))])
    out = flip_scene(640, bx, labs)
    txt = ('可翻 -> ' + str(out[1])) if out else '❌ 整图不可翻（含 FORBID）'
    print(f'{str(labs):<40s} {txt}')

bx = np.array([[10., 20., 40., 50.]])
assert flip_scene(640, bx, ['turn_left'])[1] == ['turn_right']
assert np.allclose(flip_scene(640, bx, ['turn_left'])[0], [[600., 20., 630., 50.]])
assert np.allclose(hflip_boxes(hflip_boxes(bx, 640), 640), bx), '翻两次回到原处'
assert flip_scene(640, bx, ['speed_limit_60']) is None
assert flip_scene(640, np.repeat(bx, 2, 0), ['turn_left', 'stop']) is None

# ── 反直觉的一条：对 SAFE 类，翻转的信息增益 ≈ 0 ──
def make_patch(kind, s=17):
    p = np.zeros((s, s), np.uint8); c = s // 2
    yy, xx = np.mgrid[0:s, 0:s]
    p[(xx - c) ** 2 + (yy - c) ** 2 <= c * c] = 60       # 圆形牌面
    if kind == 'no_entry':                               # 中央白横杠：左右镜像对称
        p[c - 1:c + 2, c - 5:c + 6] = 255
    else:                                                # 左向箭头：不对称
        p[c - 1:c + 2, c - 5:c + 3] = 255
        for k in range(4):
            p[c - k:c + k + 1, c - 5 + k] = 255
    return p

sym, asym = make_patch('no_entry'), make_patch('turn_left')
print(f'\n禁止驶入(SAFE)  翻转后逐像素相同? {np.array_equal(sym[:, ::-1], sym)}   '
      f'不同像素数 {int((sym[:, ::-1] != sym).sum())}')
print(f'向左转弯(SWAP)  翻转后逐像素相同? {np.array_equal(asym[:, ::-1], asym)}   '
      f'不同像素数 {int((asym[:, ::-1] != asym).sum())}')
assert np.array_equal(sym[:, ::-1], sym), 'SAFE 类的定义就是左右镜像对称 -> 翻了等于没翻'
assert not np.array_equal(asym[:, ::-1], asym)
print('\n⚠️  **在 TSR 里，翻转要么不安全（FORBID/SWAP），要么安全但没用（SAFE）。**')
print('    SAFE 的定义就是外观左右对称 -> 镜像后图块逐像素不变，模型从目标身上学到的东西')
print('    一点没变，唯一变的是背景上下文。这就是「默认关掉水平翻转」的完整理由。')

In [ ]:
# ── 量化：配置里写 fliplr=0.5，实际到底生效多少？──
FREQ = {   # TT100K 风格的长尾频率（限速牌 + 指路牌是绝对头部）
    'speed_limit_60': .16, 'guide_sign': .14, 'speed_limit_30': .09, 'ped_crossing': .08,
    'speed_limit_120': .04, 'children': .03, 'stop': .02, 'merge_left': .02,
    'no_entry': .05, 'traffic_light_ahead': .05, 'no_vehicles': .04, 'no_honking': .03,
    'turn_left': .03, 'turn_right': .03, 'no_left_turn': .025, 'no_right_turn': .025,
    'curve_left': .02, 'curve_right': .02, 'keep_left': .015, 'keep_right': .015,
    'narrow_left': .0125, 'narrow_right': .0125,
}
tot = sum(FREQ.values())
share = {k: collections.Counter() for k in ['SAFE', 'SWAP', 'FORBID']}
p_kind = collections.Counter()
for k, f_ in FREQ.items():
    p_kind[TABLE_[k]['kind']] += f_ / tot
print(f"目标级占比：SAFE {p_kind['SAFE']:.1%} | SWAP {p_kind['SWAP']:.1%} | "
      f"FORBID {p_kind['FORBID']:.1%}")

N_PER_IMG = {1: .45, 2: .28, 3: .15, 4: .08, 5: .04}       # 每图标志数分布
p_ok_obj = 1 - p_kind['FORBID']
p_img = sum(w * p_ok_obj ** n for n, w in N_PER_IMG.items())
P_FLIP_CFG = 0.5
eff_img = p_img * P_FLIP_CFG
eff_info = p_img * P_FLIP_CFG * p_kind['SWAP']             # 只有 SWAP 类真正带来新信息

print(f'\n单个目标可翻的概率            {p_ok_obj:.1%}')
print(f'**整张图**可翻的概率（每图 1-5 个标志，全都得可翻）  {p_img:.1%}')
print(f'配置写 fliplr={P_FLIP_CFG} -> 实际被翻的图         {eff_img:.1%}')
print(f'其中真正带来新信息的目标（SWAP 类）占全部目标  **{eff_info:.1%}**')

assert .55 < p_kind['FORBID'] < .65
assert .20 < p_img < .27, p_img
assert eff_info < .05, '真正有效的翻转不到 5%'
print('\n⚠️  你配置里写的 fliplr=0.5，真正被翻的图只有约 12%；')
print('    而其中真正带来新信息的目标只占全部目标的 2.5% —— **这个增强基本等于没开**，')
print('    却让你误以为「已经做了数据增强」。')
print('✅ SWAP 档还有一个附带好处：向左/向右类往往左右样本数不均衡，')
print('   SWAP 式翻转正好把两边拉平 —— 这是一个免费的类别平衡手段（面试加分点）。')

## 7 · RandomIoUCrop：小目标的最直接解法，也是位置先验的杀手

裁剪家族有两个方向相反的变体，很多人把它们混为一谈：
**RandomCrop（zoom-in）放大目标**、**Expand / Mosaic（zoom-out）缩小目标**。

zoom-in 是小目标最直接的解法（直接提高目标的相对尺寸），
但它会**摧毁位置先验**——交通标志在真实图像里几乎从不出现在画面下半部。

In [ ]:
def random_iou_crop(boxes, W, H, g, min_iou=0.3, tries=60, lo=0.3, hi=0.9,
                    y0_max=None, ch_lo=None, ch_hi=None):
    for _ in range(tries):
        s = g.uniform(lo, hi)
        cw = int(W * s)
        ch = int(H * s) if ch_lo is None else int(g.uniform(ch_lo, ch_hi) * H)
        if cw < 8 or ch < 8 or cw > W or ch > H:
            continue
        x0 = int(g.integers(0, W - cw + 1))
        y0 = int(g.integers(0, (H - ch + 1) if y0_max is None else min(y0_max, H - ch) + 1))
        patch = np.array([[x0, y0, x0 + cw, y0 + ch]], float)
        if len(boxes) == 0 or box_iou(np.asarray(boxes, float), patch).max() >= min_iou:
            return patch[0]
    return np.array([0., 0., float(W), float(H)])

W0, H0, IN = 1920, 1080, 640
px_full = sign_px(60)                                      # 60 m 外的标志，全分辨率像素
print(f'60 m 外的 0.6 m 标志，全分辨率 {px_full:.1f} px')
print(f'  直接 letterbox 到 {IN}          -> {px_full*IN/W0:>5.1f} px   (缩小 {IN/W0:.2f}x)')
for cw in [960, 640, 480]:
    print(f'  先裁 {cw}x{cw} 再 resize 到 {IN} -> {px_full*IN/cw:>5.1f} px   (放大 {IN/cw:.2f}x)')
gain = (px_full * IN / 480) / (px_full * IN / W0)
assert abs(gain - 4.0) < 1e-9, '裁 480 再放大，相对直接 letterbox 是 4 倍线性尺寸'
print(f'\n✅ 裁 480x480 再 resize，目标线性尺寸是直接 letterbox 的 **{gain:.0f} 倍** ——')
print('   小目标之所以难，就是它在特征图上占的格子太少（stride 32 时 8px 目标只占 0.25 格）。')
print('   zoom-in 直接把这个数按面积平方提上去，这是最便宜的小目标解法。')

In [ ]:
# ── 代价：位置先验被摧毁 ──
g2 = np.random.default_rng(11)
N = 20000
yc = g2.uniform(0.15, 0.45, N) * H0              # 真实：标志集中在画面上部（路侧/龙门架）

def crop_and_renorm(y0_max, ch_lo, ch_hi):
    ch = (g2.uniform(ch_lo, ch_hi, N) * H0).astype(int)
    hi_ = np.minimum(y0_max * H0, H0 - ch)
    y0 = (g2.random(N) * np.maximum(hi_, 0)).astype(int)
    inside = (yc >= y0) & (yc < y0 + ch)
    return (yc[inside] - y0[inside]) / ch[inside], ch[inside].mean()

before = yc / H0
free, ch_free = crop_and_renorm(1.0, 0.25, 0.50)          # 无约束：窗口位置全图均匀（zoom-in 用的小窗口）
cons, ch_cons = crop_and_renorm(0.05, 0.70, 1.00)         # 加约束：贴着画面上沿、窗口更大

def outside_prior(a):                            # 真实先验带是 [0.15, 0.45]
    return float(((a < 0.15) | (a > 0.45)).mean())

print(f"{'策略':<22s} {'归一化纵坐标':>16s} {'落在下半部':>11s} {'跳出先验带':>11s} {'放大倍数':>10s}")
rows = [('原图（真实分布）', before, IN / H0),
        ('无约束随机裁剪',   free,  IN / ch_free),
        ('加约束裁剪(上部)', cons,  IN / ch_cons)]
for name, arr, zoom in rows:
    print(f'{name:<22s} {arr.mean():>7.3f} ± {arr.std():<6.3f} {(arr > 0.5).mean():>10.1%} '
          f'{outside_prior(arr):>11.1%} {zoom:>9.2f}x')

p_before, p_free, p_cons = (before > .5).mean(), (free > .5).mean(), (cons > .5).mean()
o_free, o_cons = outside_prior(free), outside_prior(cons)
assert p_before == 0.0 and outside_prior(before) == 0.0, '真实世界里标志不出现在画面下半部'
assert p_free > 0.30 and o_free > 0.50, '无约束裁剪把位置先验彻底拉平'
assert p_cons < 0.12 and o_cons < 0.25, '加约束后基本保住了位置先验'
assert IN / ch_free > IN / ch_cons > IN / H0, '放大倍数：自由裁剪 > 受约束裁剪 > 直接 letterbox'
print(f'\n⚠️  现实中「标志出现在画面下半部」的概率是 **0%**；')
print(f'    无约束随机裁剪把它变成 **{p_free:.0%}**，并让 **{o_free:.0%}** 的标志跳出真实先验带 ——')
print('    模型因此失去了压制广告牌 / 车身贴纸误检的最强线索（C55 模块 03 的一个独立失效模式）。')
print(f'    加约束（窗口贴着画面上沿、窗口更大）把它压回 {p_cons:.0%} / {o_cons:.0%}，')
print(f'    同时仍保留 {IN/ch_cons:.2f}x 的放大（直接 letterbox 只有 {IN/H0:.2f}x）。')
print(f'    代价是放大倍数从 {IN/ch_free:.2f}x 降到 {IN/ch_cons:.2f}x —— **这就是那个权衡的具体数字。**')
print('\n✅ TSR 裁剪配方：① 限制窗口纵向范围保住位置先验；② 用 IoU 约束保证至少一个目标完整；')
print('   ③ 被切断的目标一律 ignore 而非 drop；④ 裁剪与 zoom-out 成对使用让尺度双向展开。')

## ✏️ 练习 1：`warp_boxes_vis`

实现 `warp_boxes_vis(boxes, M, W, H)`，返回 `(clipped, vis)`：

1. 用**四角法**把 `boxes` 经 `M` 变换（可以直接调用已有的 `warp_boxes`）；
2. 把结果裁到 `[0,W] × [0,H]`；
3. `vis` = 裁剪后面积 / 裁剪前面积（变换后面积为 0 时记 0.0）。

In [ ]:
def warp_boxes_vis(boxes, M, W, H):
    # TODO: 四角法 -> 裁剪 -> 可见比例
    #       返回 (clipped: (N,4) float, vis: (N,) float)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
B1 = np.array([[0., 0., 40., 40.]])
c, v = warp_boxes_vis(B1, M_T(600, 0), 640, 640)
assert np.allclose(c, [[600., 0., 640., 40.]]) and abs(v[0] - 1.0) < 1e-9
c, v = warp_boxes_vis(B1, M_T(620, 0), 640, 640)
assert np.allclose(c, [[620., 0., 640., 40.]]) and abs(v[0] - 0.5) < 1e-9, v
c, v = warp_boxes_vis(B1, M_T(700, 0), 640, 640)
assert abs(v[0]) < 1e-12, '完全出画面 -> vis=0'

# 旋转 45 度：框膨胀到 2 倍，但仍完全在画面内 -> vis=1
c, v = warp_boxes_vis(np.array([[300., 300., 340., 340.]]), about(M_R(45), 320, 320), 640, 640)
assert abs(box_area(c)[0] / 1600 - 2.0) < 1e-6, '45 度旋转 -> 面积翻倍'
assert abs(v[0] - 1.0) < 1e-9, 'vis 衡量的是「越界损失」，不是「膨胀」'
print('四角法+裁剪的可见比例：', np.round(v, 3))
print('45° 旋转后的框：', np.round(c[0], 1), ' 面积', box_area(c)[0], '(原 1600)')
print('✅ 练习 1 通过：注意 vis 只衡量越界损失，**框膨胀是另一码事，vis 看不见它**。')

## ✏️ 练习 2：letterbox 逆变换

实现 `letterbox_inverse(boxes_lb, r, left, top)`：把 letterbox 坐标系里的框
还原成原图坐标。**这就是推理端每一帧都要跑的那几行**，也是 C60 模块 04 里
「框整体偏移」的头号成因。

In [ ]:
def letterbox_inverse(boxes_lb, r, left, top):
    # TODO: x = (x' - left)/r ;  y = (y' - top)/r
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
p = letterbox_params((1080, 1920), (640, 640))
true = np.array([[948., 528., 972., 552.], [100., 200., 140., 260.]])
lbv = lb_fwd(true, p)
rec = letterbox_inverse(lbv, p['r'], p['left'], p['top'])
assert np.allclose(rec, true), f'往返必须无损，实得 {rec}'

bad = letterbox_inverse(lbv, p['r'], 0.0, 0.0)          # 忘记减 padding
dy = float(abs(bad[0, 1] - true[0, 1]))
assert abs(dy - 420.0) < 0.5, f'忘记减 pad 应偏 420 px，实得 {dy}'
assert box_iou(bad[:1], true[:1])[0, 0] == 0.0

# 不放大小图：scaleup=False 时 r 被夹到 1.0
p_small = letterbox_params((320, 480), (640, 640))
assert p_small['r'] == 1.0, '推理端默认不放大小图'
p_up = letterbox_params((320, 480), (640, 640), scaleup=True)
assert abs(p_up['r'] - 640 / 480) < 1e-9

print(f'正确逆变换 -> {np.round(rec[0],1)}   ✅')
print(f'忘记减 pad -> {np.round(bad[0],1)}   ❌ y 偏 {dy:.0f} px，IoU=0')
print('✅ 练习 2 通过：**把 (r,left,top) 当元数据传下去，不要在推理端重算。**')

## ✏️ 练习 3：翻转安全性检查器

实现 `safe_hflip(W, boxes, labels, table)`：

- 若任一 label 的档位是 `FORBID` → 返回 `None`（整图不能翻）；
- 否则返回 `(new_boxes, new_labels)`：框做水平翻转，
  `SWAP` 档的 label 换成它的镜像类，`SAFE` 档保持不变。

In [ ]:
def safe_hflip(W, boxes, labels, table):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
BB = np.array([[10., 20., 40., 50.], [100., 20., 130., 50.]])
out = safe_hflip(640, BB, ['turn_left', 'no_entry'], TABLE_)
assert out is not None
nb, nl = out
assert nl == ['turn_right', 'no_entry'], nl
assert np.allclose(nb, [[600., 20., 630., 50.], [510., 20., 540., 50.]]), nb
assert safe_hflip(640, BB, ['turn_left', 'speed_limit_60'], TABLE_) is None
assert safe_hflip(640, BB[:1], ['merge_left'], TABLE_) is None, 'Q3 不过 -> FORBID'
assert safe_hflip(640, BB, ['no_entry', 'no_honking'], TABLE_)[1] == ['no_entry', 'no_honking']
back = safe_hflip(640, safe_hflip(640, BB, ['turn_left', 'no_entry'], TABLE_)[0],
                  ['turn_right', 'no_entry'], TABLE_)
assert np.allclose(back[0], BB) and back[1] == ['turn_left', 'no_entry'], '翻两次回到原状态'
print('翻转前 ', ['turn_left', 'no_entry'], np.round(BB[:, 0], 0).tolist())
print('翻转后 ', nl, np.round(nb[:, 0], 0).tolist())
print('✅ 练习 3 通过：**翻转 + 换标签**，而且翻两次能完整回到原状态（含标签）。')

## ✏️ 练习 4：从丢弃阈值反推最大检出距离

实现 `range_from_min_side(min_side_px, input_w, sensor_w, hfov_deg, sign_m)`：
用针孔模型算出「配置里这个 `min_side` 把系统的最大检出距离锁死在多少米」。

`f = sensor_w / (2·tan(hfov/2))`；标志在输入分辨率下的像素尺寸
`= f · sign_m / Z · (input_w / sensor_w)`；令它等于 `min_side_px` 解出 `Z`。

In [ ]:
def range_from_min_side(min_side_px, input_w=640, sensor_w=1920, hfov_deg=60.0, sign_m=0.6):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
assert abs(range_from_min_side(8) - 41.57) < 0.05, range_from_min_side(8)
assert abs(range_from_min_side(2) - 166.28) < 0.05
assert abs(range_from_min_side(4) - 2 * range_from_min_side(8)) < 1e-9, '距离与阈值成反比'
# 换一颗长焦相机（30° FOV）：同样的 min_side 能看得更远
assert range_from_min_side(8, hfov_deg=30) > 2 * range_from_min_side(8, hfov_deg=60)
# 提高输入分辨率同样有效
assert abs(range_from_min_side(8, input_w=1280) - 2 * range_from_min_side(8, input_w=640)) < 1e-9

print(f"{'配置':<40s} {'最大检出距离':>13s} {'120km/h 反应时间':>18s}")
for desc, kw in [('640 输入 / 60° 广角 / min_side=8', dict(min_side_px=8)),
                 ('640 输入 / 60° 广角 / min_side=2', dict(min_side_px=2)),
                 ('1280 输入 / 60° 广角 / min_side=2', dict(min_side_px=2, input_w=1280)),
                 ('640 输入 / 30° 长焦 / min_side=2', dict(min_side_px=2, hfov_deg=30))]:
    Z = range_from_min_side(**kw)
    print(f'{desc:<40s} {Z:>11.1f} m {Z/(120/3.6):>16.1f} s')
print('\n✅ 练习 4 通过：**增强流水线里的一个整数，等价于一条硬件级的能力上限。**')
print('   面试里被问「怎么提升远距离检出」，这三个旋钮（min_side / 输入分辨率 / 焦距）')
print('   都是可以量化回答的 —— 而第一个是免费的。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def warp_boxes_vis(boxes, M, W, H):
    w = warp_boxes(boxes, M)
    a0 = box_area(w)
    c = w.copy()
    c[:, [0, 2]] = np.clip(c[:, [0, 2]], 0, W)
    c[:, [1, 3]] = np.clip(c[:, [1, 3]], 0, H)
    a1 = box_area(c)
    vis = np.where(a0 > 0, a1 / np.maximum(a0, 1e-12), 0.0)
    return c, vis

In [ ]:
# 练习 2 参考答案
def letterbox_inverse(boxes_lb, r, left, top):
    b = np.asarray(boxes_lb, float).reshape(-1, 4).copy()
    b[:, [0, 2]] = (b[:, [0, 2]] - left) / r
    b[:, [1, 3]] = (b[:, [1, 3]] - top) / r
    return b

In [ ]:
# 练习 3 参考答案
def safe_hflip(W, boxes, labels, table):
    if any(table[c]['kind'] == 'FORBID' for c in labels):
        return None
    new_labels = [table[c]['mirror'] if table[c]['kind'] == 'SWAP' else c for c in labels]
    return hflip_boxes(boxes, W), new_labels

In [ ]:
# 练习 4 参考答案
def range_from_min_side(min_side_px, input_w=640, sensor_w=1920, hfov_deg=60.0, sign_m=0.6):
    f = sensor_w / (2 * math.tan(math.radians(hfov_deg) / 2))
    return f * sign_m * (input_w / sensor_w) / min_side_px

---
## 🧪 真实工程胶囊：一份可直接用的几何增强配置与验收清单

下面这段可以原样复制进项目（把 numpy 换成 OpenCV / albumentations 即可）。
它的价值在于：**每一个数字都有出处，不是抄来的。**

In [ ]:
RECIPE = r'''
# ============================================================
# TSR 几何增强配置（configs/aug_geometric.yaml）—— 每个数字都写清出处
# ============================================================
geometric:
  # 1) 缩放：TSR 收益最高的几何增强。对应「标志的远近」，物理上精确无误差。
  random_scale:   {range: [0.5, 1.5], p: 1.0}

  # 2) 平移：TSR 有强位置先验（标志在画面上部/路侧），幅度不能大。
  random_translate: {frac: 0.10, p: 0.5}

  # 3) 旋转：**上限由「可接受的框膨胀率」反推**，不是拍脑袋。
  #    A'/A = 1 + 0.5*|sin2θ|*(w/h + h/w)
  #    可接受膨胀 <=35%  ->  正方形框 |θ|<=10.2° ; 3:1 指路牌 |θ|<=6.1°
  #    取较小者，且真实车身滚转角 p99 约 3-5°，故：
  random_rotate:  {deg: 6.0, p: 0.3}

  # 4) 错切/透视：与轻微旋转作用重叠，且透视容易失控。默认关闭。
  random_shear:       {deg: 0.0}
  random_perspective: {scale: 0.0}     # 开启时务必检查角点变换做了透视除法

  # 5) 水平翻转：**TSR 默认关闭**。要开必须配三档白名单（见下）。
  #    量化过：整图可翻率 ~23%，配 p=0.5 实际只翻 ~12% 的图，
  #    其中真正带来新信息的（SWAP 类）只占全部目标的 ~2.5%。
  random_hflip:   {p: 0.0, whitelist: configs/flip_whitelist.yaml}
  random_vflip:   {p: 0.0}             # 永远关闭：重力方向是恒定的

  # 6) 裁剪：小目标最直接的解法，但会摧毁位置先验。**必须加约束**。
  random_iou_crop:
    min_iou_choices: [0.1, 0.3, 0.5, 0.7, 0.9]
    crop_y0_max_frac: 0.05             # 窗口贴着画面上沿 -> 保住位置先验
    crop_h_frac: [0.70, 1.00]          # 窗口不能太小，否则位置分布被拉平
    # 实测：无约束裁剪会让「标志落在画面下半部」的概率从 0% 涨到 ~45%

# ============================================================
# 越界策略：**只有完全出画面才 drop，其余一律 ignore**
# ============================================================
out_of_bound:
  min_visible: 0.5                     # < 0.5 且 > 0 -> ignore（不是 drop！）
  min_side_px: 2                       # **这个数等价于最大检出距离**：
                                       #   min_side=8 -> 41.6 m（120km/h 只剩 1.2 s）
                                       #   min_side=2 -> 166 m
  policy_below_threshold: ignore       # 绝不用 drop：会制造假负样本监督

# ============================================================
# letterbox：全流水线唯一必须与推理端逐像素一致的算子
# ============================================================
letterbox:
  size: [640, 640]
  pad_value: 114
  center: true
  scaleup: false
  auto_stride: null                    # **导出 ONNX/TRT 时必须为 null**，否则输出不是 640x640
  rounding: round                      # round vs floor 差半个 pad 像素 -> 原图 1.5 px
                                       #   -> 8px 标志 IoU 从 1.00 掉到 0.49
  # 逆变换：把 (r, left, top) 作为**元数据**传给推理端，不要重算。
  #   x = (x_lb - left) / r ;  y = (y_lb - top) / r
  #   对应 YOLOv5: scale_boxes(img1_shape, boxes, img0_shape, ratio_pad=(r,(left,top)))

# ============================================================
# 验收清单（每个几何算子合入前必须过）
# ============================================================
# [ ] apply_image 与 apply_boxes 用的是**同一个矩阵 M**
# [ ] 角点变换**无条件**做透视除法（仿射时是恒等操作）
# [ ] 用目标掩码/轮廓算出的真实紧框做断言，IoU >= 0.95（不要靠可视化抽查）
# [ ] 越界目标走 ignore 而非 drop，且单测覆盖「露一半」的情况
# [ ] letterbox forward -> inverse 往返误差 < 0.5 px（含奇数尺寸、非对齐尺寸）
# [ ] 翻转走白名单，且「翻两次回到原状态（含标签）」有单测
# [ ] 增强前后的目标尺寸直方图对比，确认小尺寸桶没有整段消失
'''
print(RECIPE)
for k in ['random_scale', 'random_rotate', 'whitelist', 'min_side_px', 'ignore',
          'auto_stride', 'ratio_pad', 'crop_y0_max_frac', '透视除法']:
    assert k in RECIPE, k
print('✅ 配方覆盖：缩放/平移/旋转(反推上限)/翻转白名单/受约束裁剪/越界策略/'
      'letterbox 正逆变换/验收清单')

### 小结

- **几何增强的唯一正确形态是「一个矩阵」**：只插值一次、绕某点变换有明确写法、
  图像与标注用同一个对象所以不可能不同步。`T(c)·M·T(-c)` 这个三明治结构要背下来。
- **角点变换必须无条件做透视除法**。仿射时它是恒等操作，透视时它是必需的——
  幅度小的时候误差也小，所以这个 bug 能潜伏很久。
- **旋转会让外接框膨胀**：`A'/A = 1 + ½|sin2θ|(w/h + h/w)`。
  **正方形框 45° 时面积翻倍、15° 时已膨胀 50%**；细长的指路牌更狠（3:1 时 45° 膨胀 2.67×）。
  **旋转幅度上限应从「可接受的膨胀率」反推**，而不是拍脑袋。
- **四角法对旋转不变的形状是错的**：圆形禁令牌旋转 45°，真实紧框一点没变，
  四角法却把框撑到 √2 倍边长——**标注 IoU 只剩 0.50、框内前景只剩 39%**。
  这不是「任务变难」，是「标签变错」。
- **越界三分法**：`keep` 说「有」、`ignore` 说「不知道」、`drop` 说「没有」。
  **只有完全出画面才 drop**；否则可见的目标被删就成了明确的假负样本监督——
  而「标志被画面边缘截断」是 TSR 每次接近标志的必经阶段。
- **`min_side` 等价于一条硬件级能力上限**：640 输入 / 60° FOV 下，
  `min_side=8` 把最大检出距离锁死在 **41.6 m**（120 km/h 只剩 1.2 秒）；`min_side=2` 是 166 m。
- **letterbox 的逆变换是它存在的全部理由**。四种真实 bug：忘记减 pad（偏 420 px）、
  各轴独立缩放、stride 对齐不一致（偏 384 px）、round vs floor（只偏 1.5 px，
  但 **8 px 标志的 IoU 掉到 0.49**）。**把 (r, left, top) 当元数据传下去，不要重算。**
- **水平翻转在 TSR 里默认关闭**。三问判定 → SAFE / SWAP / FORBID；
  判定是逐目标的而翻转是整图的（「与」关系），实际生效率只有约 12%；
  而 **SAFE 类翻转后逐像素不变，信息增益为零**。
  **SWAP 档（翻转 + 换标签）是加分答案**，还顺带平衡了左右类样本数。
- **裁剪是小目标最直接的解法**（裁 480 再放大 = 4 倍线性尺寸），
  但无约束裁剪会把「标志出现在画面下半部」的概率从 **0% 推到 45%**，
  摧毁压制广告牌误检的最强线索。**要用加约束的裁剪。**

下一站：**模块 02 · 光度增强与域鲁棒性** —— HSV/gamma/噪声/模糊、
**「颜色即语义」的红线**（红=禁令、蓝=指示、黄=警告）、大气散射雾化模型、
运动模糊核与卷帘快门。